# 摘要缓冲记忆

> **一种混合记忆，保留最近消息的原文同时摘要旧历史。**

想想你如何记住与朋友的一次长电话通话。最后几句话？你几乎逐字逐句地记得。那是你的短期记忆。更早的部分？你记得大意，而不是精确的词语。那是你的长期记忆。

摘要缓冲记忆以同样的方式工作。它保留一个**缓冲区**存储最近消息的原始形式。所有更旧的内容被压缩成**运行摘要**。LLM 在每次调用时接收这两部分：摘要提供历史上下文，最近的原始消息提供精确性。

纯缓冲记忆（存储每条消息）提供精确回忆但每轮消耗更多 token。纯摘要记忆（压缩所有内容）节省 token 但丢失最近细节。摘要缓冲记忆结合了两者的优势。它对于客服机器人、辅导 Agent 和项目管理助手尤其有价值——这些 Agent 需要记住对话的完整脉络，同时对最新消息做出精确回应。

核心工程挑战是**过渡阈值**：消息从缓冲区转移到摘要的分界点。设置得太高，你浪费 token。设置得太低，你丢失重要细节。

**完成本 notebook 后你将理解：**
- 如何使用 OpenAI SDK 从头构建摘要缓冲记忆。
- 增量摘要的工作原理（用新驱逐的消息更新摘要）。
- 如何调优缓冲区大小并观察其对 token 使用的影响。
- 本技术何时有效，何时悄然失效。

## 核心概念

- **缓冲区**：最近 *k* 条消息按原文存储。这些保留精确措辞、细微差别和细节，用于即时推理。
- **运行摘要**：一个捕获所有已移出缓冲区的消息中关键事实和主题的简短段落。LLM 生成此摘要。
- **上下文窗口**：模型在单次调用中能读取的最大 token（模型内部使用的词片）数。例如，GPT-4o 支持最多 128k token。
- **过渡阈值**：消息从缓冲区移入摘要的触发点。你可以基于消息数量或 token 数量。
- **增量摘要**：不是每次重新摘要完整历史，而是取现有摘要和新驱逐的消息，通过单次 LLM 调用生成更新的摘要。这保持了较低的摘要成本。
- **Token 预算分配**：可用上下文窗口 token 在摘要和缓冲区之间的分配。典型分配为摘要保留 20-30%，最近消息保留 70-80%。
- **提示词组裝**：将 `[system prompt] + [summary] + [recent buffer messages]` 拼接成一个提示词的过程。

## 架构

<p align="center">
  <img src="../../images/diagrams/04_summary_buffer_memory.svg" alt="摘要缓冲记忆架构图" width="720"/>
</p>

<details><summary>Mermaid 源代码</summary>

```mermaid
flowchart LR
    User["用户消息"] --> Buffer["缓冲区\n(最近 K 条消息)"]
    Buffer --> ThresholdCheck{"缓冲区超过\n阈值？"}
    ThresholdCheck -- 否 --> PromptAssembly["提示词组装"]
    ThresholdCheck -- 是 --> Evict["驱逐最旧\n的消息"]
    Evict --> Summarizer["摘要器\n(LLM 调用)"]
    Summarizer --> SummaryStore["摘要存储\n(运行摘要)"]
    SummaryStore --> PromptAssembly
    Buffer --> PromptAssembly
    PromptAssembly --> LLM["LLM\n[摘要] + [缓冲区]"]
    LLM --> Response["回复"]
    Response --> Buffer
```

</details>

数据流如下：新用户消息进入缓冲区。当缓冲区超过配置的阈值时，最旧的消息被驱逐。这些被驱逐的消息送入摘要器 LLM 调用，更新运行摘要。最终提示词连接系统提示词、摘要和最近的缓冲区消息。LLM 回复，该回复返回缓冲区。每轮循环重复。

## 环境准备

安装依赖并配置 API 访问。你需要在 `.env` 文件中设置 `OPENAI_API_KEY` 环境变量。

In [ ]:
%pip install -q openai python-dotenv tiktoken

导入 OpenAI SDK、`tiktoken`（OpenAI 用于计数 token 的 tokenizer 库）和标准库辅助工具。

In [ ]:
import os
import json
import copy
from dotenv import load_dotenv
import tiktoken
from openai import OpenAI

load_dotenv()  # 从 .env 读取 OPENAI_API_KEY

assert os.getenv("OPENAI_API_KEY"), "请在 .env 文件中设置 OPENAI_API_KEY"

client = OpenAI()

# 我们将在整个过程中使用此编码器来计数 token。
# "cl100k_base" 是 GPT-4o 和 GPT-4 使用的 tokenizer。
encoder = tiktoken.get_encoding("cl100k_base")

## 实现

我们将构建一个 `SummaryBufferMemory` 类，它：
1. 在缓冲区（普通 Python 列表）中存储最近消息。
2. 维护旧历史的运行摘要。
3. 当缓冲区超过 token 阈值时驱逐最旧的消息。
4. 使用 LLM 调用将被驱逐的消息合并到摘要中。
5. 将最终提示词组装为 `[system] + [summary] + [buffer]`。

### Token 计数辅助函数

我们需要一种方法来计算消息列表使用了多少 token。这告诉我们缓冲区何时增长得太大了。

In [ ]:
def count_message_tokens(messages: list[dict], enc: tiktoken.Encoding = encoder) -> int:
    """计算聊天消息列表中的总 token 数。

    每条消息都有角色格式化的开销 token。
    我们为每条消息添加 4 个 token 作为该开销的近似值。
    """
    total = 0
    for msg in messages:
        total += 4  # 角色 + 格式化开销
        total += len(enc.encode(msg["content"]))
    return total

### 摘要器

此函数接收当前运行摘要和一批新驱逐的消息。它要求 LLM 生成一个包含新信息的更新摘要。

想象它就像更新会议记录。你手里有昨天的记录（现有摘要）和今天的讨论要点（被驱逐的消息）。你将它们合并为一份简洁的文档。

In [ ]:
SUMMARY_SYSTEM_PROMPT = """你是一个对话摘要器。你的工作是维护对话的运行摘要。你将收到：
1. 当前摘要（如果这是开始，可能为空）。
2. 需要合并的新消息。

生成一个更新后的摘要，要求：
- 保留所有关键事实、名称、偏好和决定。
- 保持简洁（目标 3-6 句话）。
- 使用第三人称（"用户说……"、"助手建议……"）。
- 去掉闲聊和填充语，保留实质性内容。
"""


def update_summary(
    current_summary: str,
    evicted_messages: list[dict],
    model: str = "gpt-4o-mini",
) -> str:
    """通过 LLM 调用将被驱逐的消息合并到运行摘要中。"""
    # 将被驱逐的消息格式化为可读文本
    evicted_text = "\n".join(
        f"{msg['role'].upper()}: {msg['content']}" for msg in evicted_messages
    )

    user_prompt = (
        f"当前摘要：\n{current_summary if current_summary else '（空）'}\n\n"
        f"需要合并的新消息：\n{evicted_text}\n\n"
        f"请编写更新后的摘要："
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=512,
        temperature=0.3,  # 低温度以获得一致的摘要
    )

    return response.choices[0].message.content

### SummaryBufferMemory 类

这是本技术的核心。该类管理两个记忆区域：
- **缓冲区**：最近消息的列表（逐字存储）。
- **摘要**：一个捕获旧历史的字符串。

当你调用 `chat()` 时，类检查缓冲区是否超过 `max_buffer_tokens`。如果超过，最旧的消息被驱逐并合并到摘要中。然后它组装完整提示词并调用 LLM。

In [ ]:
class SummaryBufferMemory:
    """混合记忆：运行摘要 + 最近消息缓冲区。"""

    def __init__(
        self,
        model: str = "gpt-4o-mini",
        system_prompt: str | None = None,
        max_buffer_tokens: int = 300,
        max_tokens: int = 1024,
    ):
        self.model = model
        self.system_prompt = system_prompt
        self.max_buffer_tokens = max_buffer_tokens
        self.max_tokens = max_tokens

        # 两个记忆区域
        self.buffer: list[dict] = []      # 最近消息（逐字存储）
        self.summary: str = ""             # 压缩后的旧历史

        # 记录统计
        self.turn_count: int = 0
        self.eviction_log: list[dict] = []  # 跟踪每次驱逐事件



接下来我们添加驱逐和提示词构建逻辑。`_evict_if_needed` 从缓冲区逐个弹出最旧的消息，然后调用 `update_summary` 将它们合并到运行摘要中。`_build_messages` 将系统提示词、摘要和剩余缓冲区拼接成一个提示词发送给 API。

In [ ]:
    def _evict_if_needed(self) -> None:
        """当超出预算时，将最旧的消息从缓冲区移至摘要。"""
        evicted_batch = []

        while (
            len(self.buffer) > 2
            and count_message_tokens(self.buffer) > self.max_buffer_tokens
        ):
            # 驱逐最旧的消息（FIFO：先进先出）
            evicted_batch.append(self.buffer.pop(0))

        if evicted_batch:
            old_summary = self.summary
            self.summary = update_summary(self.summary, evicted_batch, self.model)

            self.eviction_log.append({
                "turn": self.turn_count,
                "evicted_count": len(evicted_batch),
                "summary_before": old_summary,
                "summary_after": self.summary,
            })

    def _build_messages(self) -> list[dict]:
        """组装发送给 API 的完整消息列表。"""
        messages = []

        # 将摘要作为系统级上下文块注入
        if self.summary:
            summary_block = f"更早对话的摘要：\n{self.summary}"
            if self.system_prompt:
                messages.append({
                    "role": "system",
                    "content": f"{self.system_prompt}\n\n{summary_block}",
                })
            else:
                messages.append({"role": "system", "content": summary_block})
        elif self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})

        # 追加最近的缓冲区消息
        messages.extend(self.buffer)
        return messages



`chat` 方法将所有内容串联起来。它追加用户消息，必要时触发驱逐，构建组装好的提示词，并调用 LLM。回复返回缓冲区以供未来上下文使用。

In [ ]:
    def chat(self, user_input: str) -> str:
        """发送消息并获取回复。"""
        self.buffer.append({"role": "user", "content": user_input})
        self.turn_count += 1

        # 在调用 LLM 之前检查是否需要驱逐
        self._evict_if_needed()

        # 构建提示词并调用 API
        messages = self._build_messages()

        response = client.chat.completions.create(
            model=self.model,
            messages=messages,
            max_tokens=self.max_tokens,
        )

        assistant_text = response.choices[0].message.content
        self.buffer.append({"role": "assistant", "content": assistant_text})

        return assistant_text



最后，我们添加用于检查和重置记忆的工具方法。`get_buffer` 和 `get_summary` 让你查看每个区域。`get_buffer_token_count` 告诉你缓冲区距离驱逐阈值有多近。

In [ ]:
    def get_buffer(self) -> list[dict]:
        """返回当前缓冲区的副本。"""
        return copy.deepcopy(self.buffer)

    def get_summary(self) -> str:
        """返回当前运行摘要。"""
        return self.summary

    def get_buffer_token_count(self) -> int:
        """返回缓冲区当前的 token 数。"""
        return count_message_tokens(self.buffer)

    def clear(self) -> None:
        """重置两个记忆区域。"""
        self.buffer.clear()
        self.summary = ""
        self.turn_count = 0
        self.eviction_log.clear()

    def __repr__(self) -> str:
        return (
            f"SummaryBufferMemory("
            f"buffer={len(self.buffer)} msgs, "
            f"summary={'有' if self.summary else '空'}, "
            f"turns={self.turn_count})"
        )

## 示例运行

让我们运行一个多轮对话并观察摘要缓冲的实际运作。我们将使用一个小的缓冲区阈值（300 token），以便驱逐快速发生。在生产中你会设置得更高。

创建记忆并运行对话。我们将在每轮后打印 Agent 的回复。

In [ ]:
memory = SummaryBufferMemory(
    system_prompt="你是一个有帮助、简洁的助手。回复请控制在两句话以内。",
    max_buffer_tokens=300,
)

exchanges = [
    "你好！我叫 Alice，是柏林的一名机器学习工程师。",
    "我正在为我们的客服团队构建一个聊天机器人。我们每天大约有 500 张工单。",
    "主要问题是我们目前的机器人在 3 条消息后就会忘记上下文。",
    "我们后端使用 Python 和 FastAPI。机器人运行在 GPT-4o 上。",
    "我们的客户主要询问账单、配送和退货。",
    "我还希望机器人能跨会话记住用户偏好。",
    "我们目前在 PostgreSQL 中存储对话日志。",
    "你会为我们的用例推荐哪种记忆技术？",
    "顺便问一下，我叫什么名字，在哪里工作？",
]

for msg in exchanges:
    print(f"\n{'='*60}")
    print(f"第 {memory.turn_count + 1} 轮")
    print(f"{'='*60}")
    print(f"用户：  {msg}")
    reply = memory.chat(msg)
    print(f"Agent：{reply}")
    print(f"\n  缓冲区：{len(memory.buffer)} 条消息（{memory.get_buffer_token_count()} token）")
    print(f"  摘要：{'[空]' if not memory.summary else memory.summary[:100] + '...'}")

### 检查记忆状态

经过 9 轮后，一些消息已被驱逐并摘要。让我们查看每个区域中的内容。

In [ ]:
print("运行摘要")
print("-" * 40)
print(memory.get_summary() or "（空）")

print("\n缓冲区（最近消息）")
print("-" * 40)
for i, msg in enumerate(memory.get_buffer()):
    role_label = "用户" if msg["role"] == "user" else "助手"
    preview = msg["content"][:90] + ("..." if len(msg["content"]) > 90 else "")
    print(f"  [{i}] {role_label}: {preview}")

print(f"\n缓冲区 token 数：{memory.get_buffer_token_count()}")
print(f"缓冲区阈值：      {memory.max_buffer_tokens}")

### 驱逐日志

每次消息移出缓冲区时，系统记录该事件。此日志显示摘要如何随时间演化。

In [ ]:
print(f"驱逐事件总数：{len(memory.eviction_log)}\n")

for i, event in enumerate(memory.eviction_log):
    print(f"驱逐 {i + 1}（第 {event['turn']} 轮后）")
    print(f"  驱逐消息数：{event['evicted_count']}")
    print(f"  驱逐前摘要：{event['summary_before'][:80] or '（空）'}...")
    print(f"  驱逐后摘要：{event['summary_after'][:80]}...")
    print()

### LLM 实际看到的内容

让我们偷看组装好的提示词。这正是下一次调用时发送给 API 的内容。注意摘要位于系统消息中，缓冲区消息紧随其后。

In [ ]:
assembled = memory._build_messages()

print(f"组装提示词中的消息总数：{len(assembled)}")
print(f"组装提示词中的 token 总数：   {count_message_tokens(assembled)}\n")

for i, msg in enumerate(assembled):
    role = msg["role"].upper()
    content = msg["content"]
    if len(content) > 150:
        content = content[:150] + "..."
    print(f"[{i}] {role}:")
    print(f"    {content}")
    print()

## 对比：缓冲 vs. 摘要缓冲

我们的混合方案在 token 使用方面与普通缓冲记忆相比如何？让我们在两者上运行相同的对话并测量。

In [ ]:
# 模拟普通缓冲记忆：计算每轮发送所有消息时的 token 数。
# 我们复用上面相同的 exchanges 列表。

all_messages = []  # 像普通缓冲区一样累积
buffer_tokens_per_turn = []
summary_buffer_tokens_per_turn = []

# 重置摘要缓冲记忆以进行干净的运行
sbm = SummaryBufferMemory(
    system_prompt="你是一个有帮助、简洁的助手。回复请控制在两句话以内。",
    max_buffer_tokens=300,
)

for msg in exchanges:
    # 普通缓冲：累积所有消息
    all_messages.append({"role": "user", "content": msg})
    buffer_tokens_per_turn.append(count_message_tokens(all_messages))
    # 添加占位助手回复以保持计数真实
    all_messages.append({"role": "assistant", "content": "已收到。"})

    # 摘要缓冲：使用实际的类
    sbm.chat(msg)
    assembled_msgs = sbm._build_messages()
    summary_buffer_tokens_per_turn.append(count_message_tokens(assembled_msgs))

print(f"{'轮次':<6} {'缓冲记忆':<18} {'摘要缓冲记忆'}")
print("-" * 48)
for i in range(len(exchanges)):
    print(f"{i+1:<6} {buffer_tokens_per_turn[i]:<18} {summary_buffer_tokens_per_turn[i]}")

print(f"\n最终轮次 token 使用量：")
print(f"  缓冲记忆：         {buffer_tokens_per_turn[-1]} token")
print(f"  摘要缓冲记忆：     {summary_buffer_tokens_per_turn[-1]} token")
print(f"  节省：             {buffer_tokens_per_turn[-1] - summary_buffer_tokens_per_turn[-1]} token")

## 权衡

### 摘要缓冲记忆适用场景

- **中长对话**（20-100+ 轮），需要历史上下文和精确的最近细节。客服机器人和辅导 Agent 是常见例子。
- **注重 token 预算的应用**：摘要压缩旧历史，因此累积 token 使用量比纯缓冲记忆增长慢得多。
- **当最近上下文最重要时**：缓冲区保留最后几次对话的全部忠实度。模型可以引用最近轮次中的确切词语、数字和名称。

### 失效场景

- **摘要质量随时间退化。** 每次增量摘要调用都有可能丢失细节。经过数十次驱逐后，摘要可能会遗漏重要事实。没有内置的方式来验证摘要的准确性。
- **额外的 LLM 调用增加成本和延迟。** 每次驱逐都会触发一次摘要调用。对于快节奏对话（亚秒级响应要求），这种开销可能太高。
- **比纯缓冲更难调试。** 当 Agent 遗忘某事时，你需要追踪信息是在摘要期间丢失还是从未到达缓冲区。使用纯缓冲记忆，完整历史始终存在。
- **调优很棘手。** 缓冲区阈值、摘要提示词和摘要模型的选择都会相互影响。找到合适的平衡需要实验。

## 进一步阅读

- [LangChain ConversationSummaryBufferMemory](https://python.langchain.com/docs/modules/memory/types/summary_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：本模式的框架实现，具有可配置的 token 限制和基于 LLM 的摘要。
- [MemGPT / Letta: 分层记忆管理 (arXiv:2310.08560)](https://arxiv.org/abs/2310.08560)：Packer 等人引入了一种多层记忆架构，LLM 管理自己的上下文窗口，在主上下文和外部存储之间移动数据。
- [LlamaIndex ChatSummaryMemoryBuffer](https://docs.llamaindex.ai/en/stable/api_reference/memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：LlamaIndex 的实现，将摘要缓冲记忆与检索增强生成流水线集成。
- [OpenAI 实用指南：使用 tiktoken 计数 Token](https://cookbook.openai.com/examples/how_to_count_tokens_with_tiktoken?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：在聊天应用中计数和管理 token 的实用策略。

---

*← 上一章：[03 摘要记忆](../03_summary_memory/) · 下一章：[05 Token 缓冲记忆](../05_token_buffer_memory/) →*

## 🧪 自己动手试试

三个小挑战来加深你的理解。每个应该花费 10-30 分钟。

### 挑战 1：调优驱逐阈值
将 `SummaryBufferMemory` 中的 token 阈值修改为 500、1000 和 2000 token。对每种设置运行 15 轮对话，记录 `_evict_if_needed()` 触发的频率。每次记录最终摘要长度。

### 挑战 2：压缩比追踪
每次驱逐周期后，计算节省的 token 比率（被驱逐的原始 token 减去摘要 token）。绘制此压缩比随轮次变化的曲线。找出摘要收益递减的点。

### 挑战 3：实体感知摘要
在调用 `update_summary()` 之前，使用 07 实体记忆中的方法从被驱逐的消息中提取实体。将实体列表传递给摘要提示词以保留命名实体。比较此更改前后对实体特定问题的回忆表现。

![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--04-summary-buffer-memory--summary-buffer-memory)
